# 02 · Data Analysis – Tổng hợp Q1–Q5 (và đối chiếu Q6)
**Nhóm CSC17104:** Trần Nhật Dương · Nguyễn Mạnh Hà · Hoàng Công Lý

Đây là **notebook tổng hợp và điều hướng** cho toàn bộ phần phân tích non-ML (`Q1–Q5`), liên kết kết quả thực nghiệm từ các notebook thành phần và đối chiếu với mô hình ML (`Q6`). Tất cả con số dưới đây đều trích xuất trực tiếp từ đầu ra đã chạy trên bộ dữ liệu UCI #603 (`12.684` kịch bản khảo sát).

## 1. Phân công & phạm vi tiêu chí độ sâu (D2–D6)

| Câu | Owner | Notebook | Tiêu chí đã thực thi | Số biểu đồ |
| --- | --- | --- | --- | ---: |
| **Q1** | Trần Nhật Dương | `02A_duong_q1_q2.ipynb` | **D3** (`travel_time_band`), **D4** (`destination`, `coupon`), **D5** (bootstrap 95% CI + độ nhạy trùng lặp) | 3 |
| **Q2** | Trần Nhật Dương | `02A_duong_q1_q2.ipynb` | **D2** (`pivot_table` & `melt`), **D4** (`coupon`, `time × coupon`), **D5** (bootstrap 95% CI) | 3 |
| **Q3** | Nguyễn Mạnh Hà | `02B_ha_q3_q4.ipynb` | **D3** (`matched_frequency`, `frequent_user`), **D4** (trong từng `coupon`), **D5** (Wilson + bootstrap CI), **D6** (4 phương án ngưỡng/NaN + dedup) | 2 |
| **Q4** | Nguyễn Mạnh Hà | `02B_ha_q3_q4.ipynb` | **D4** (`coupon`, `destination`, `has_children` + chuẩn hóa common support), **D5** (Wilson + bootstrap CI) | 2 |
| **Q5** | Hoàng Công Lý | `02C_ly_q5.ipynb` | **D4** (chuẩn hóa `coupon × thói quen`), **D5** (cluster-bootstrap CI theo 567 người), **D6** (đổi cách gộp tuổi/thu nhập, ngưỡng thói quen, đơn vị bootstrap) | 4 |
| **Q6** | Hoàng Công Lý | `03_ml_question.ipynb` | ML riêng: leakage audit, group split, Pipeline, `Dummy` + `LogReg` + `RF`, 5-fold CV, test 1 lần, error analysis, permutation importance | 4 |

Phạm vi non-ML bao phủ **5/6 tiêu chí (`D2, D3, D4, D5, D6`)**.

## 2. Bảng tổng hợp kết quả định lượng (Q1–Q6)

| Câu | Kết luận định lượng chính (số liệu thực) | Biểu đồ chính | Hạn chế cốt lõi |
| --- | --- | --- | --- |
| **Q1** | `5–<15 min` nhận **61,42%** (`n=5.562`) > `15–<25 min` **56,07%** (`n=5.611`) > `≥25 min` **42,89%** (`n=1.511`), chênh **+18,53 điểm %** (CI 95%: `[15,64; 21,27]`). Mức giảm mạnh ở `Restaurant(<20)` (**+42,74 pp**) và `Coffee House` (**+20,29 pp**), nhưng phẳng ở `Carry out` (**+2,05 pp**). Cùng hướng (`direction_same=1`) trong `Home` tăng **+17,40 pp** (CI `[13,92; 20,84]`), ngược lại toàn mẫu chỉ `+1,76 pp` do `No Urgent Place` luôn có `direction_same=0`. | `q1_fig1_heatmap_travel_time_direction.png`, `q1_fig2_grouped_bar_coupon_travel_time.png`, `q1_fig3_stratified_destination_direction.png` | Cờ ngưỡng rời rạc (không có km/phút GPS), ô rỗng cấu trúc (`No Urgent Place` không có `≥25 min`), chưa tách `respondent_id` khi bootstrap. |
| **Q2** | Toàn mẫu, hạn `1d` (**62,56%**, `n=7.091`) cao hơn `2h` (**49,60%**, `n=5.593`) là **+12,96 điểm %** (CI `[11,21; 14,63]`). Xu hướng `1d > 2h` rõ ở `7AM` (`+20,78 pp`), `2PM` (`+15,92 pp`), `6PM` (`+7,29 pp`), `10PM` (`+26,13 pp`). Riêng `10AM` gộp thô là `-2,01 pp` nhưng khi tách từng `coupon` thì **cả 5/5 loại đều `1d > 2h`** (`+2,41` đến `+23,02 pp` — nghịch lý Simpson do tỷ trọng `Bar`). `Bar` lúc `10PM` đảo chiều nhẹ (`-8,87 pp`). | `q2_fig1_grouped_bar_time_expiration.png`, `q2_fig2_heatmap_diff_time_coupon.png`, `q2_fig3_coupon_expiration_ci.png` | `time` chỉ là 5 khung giờ giả định (không phải chuỗi ngày), gắn chặt với `destination` (`7AM` = 100% `Work`); `Bar` thiếu `2h` ở `7AM` và `2PM`. |
| **Q3** | Người ghé địa điểm tương ứng thường xuyên (`1~3/4~8/gt8`) chấp nhận cao hơn ở **cả 5 loại coupon**: mạnh nhất ở `Bar` (**68,79%** vs **29,21%**, chênh **+39,58 pp**, CI `[35,09; 44,11]`) và `Coffee House` (**66,02%** vs **34,62%**, **+31,41 pp**, CI `[28,37; 34,36]`), tiếp theo `Restaurant(20-50)` (**+17,15 pp**), `Carry out` (**+5,80 pp**), `Restaurant(<20)` (**+5,39 pp**). Cả 4 phương án ngưỡng/NaN đều giữ dấu dương; bỏ 74 dòng trùng chỉ đổi tối đa `0,29 pp`. | `ha_q3_frequency_bars.png`, `ha_q3_frequency_heatmap.png` | Tần suất tự khai báo theo khoảng, có 164 dòng thiếu thói quen phù hợp (`1,29%`), không tăng đơn điệu tuyệt đối ở `4~8` vs `gt8`. |
| **Q4** | Gộp thô, `Friend(s)` cao hơn `Alone` **+14,76 pp** (CI `[12,81; 16,70]`), `Partner` cao hơn `Alone` **+6,95 pp**. Sau chuẩn hóa theo `coupon × destination` trên vùng common support (`n≥30` cả hai nhóm), chênh lệch thu hẹp còn **+1,33 pp** (CI `[-1,78; 4,48]`, `4/15` tầng, `41,6%` mẫu cặp) và **+3,03 pp** (CI `[-1,11; 7,08]`). Nguyên nhân chính: `Friend(s)` chỉ xuất hiện ở `No Urgent Place` (`n=3.298`) và vắng mặt hoàn toàn (`n=0`) ở `Home`/`Work`. Trong `Bar` tại `No Urgent Place`, `Friend(s)` vẫn cao hơn `Alone` **+17,99 pp** (CI `[10,36; 25,67]`). | `ha_q4_passanger_bars.png`, `ha_q4_destination_heatmaps.png` | Mất cân bằng hỗ trợ theo `destination` (`Friend(s)` không có ở `Home`/`Work`); `6/12` tầng đủ mẫu đổi dấu so với mức gộp; `has_children` không đo việc có trẻ đi cùng xe lúc đó. |
| **Q5** | Trên `567` người trả lời giả định (`12.520` dòng đủ thói quen): chênh lệch tuổi `<26 − 50+` thô là **+9,7 pp** (CI `[4,5; 15,0]`), tập trung ở `Bar` (**+19,6 pp**), `Coffee House` (**+13,2 pp**), `Restaurant(<20)` (**+10,3 pp**); giảm xuống **+7,5 pp** (CI `[2,8; 12,4]`) sau chuẩn hóa `coupon × thói quen`. Thu nhập `Thấp − Cao` chênh nhẹ **+4,4 pp** (CI `[0,4; 8,4]`), nhưng thu hẹp còn **+3,8 pp** (CI `[-0,6; 7,8]`) khi cố định tuổi; nhóm `≥100k` nhận **57,8%**, không giảm tuyến tính. | `q5_fig1_income_coupon_heatmap.png`, `q5_fig2_age_group_by_coupon.png`, `q5_fig3_contrasts_raw_vs_adjusted.png`, `q5_fig4_original_levels.png` | `respondent_id` suy ra từ hồ sơ trùng; `9/40` ô 3 chiều theo tuổi có `<30` người; độ lớn chênh lệch nhạy với cách chia nhóm & ngưỡng thói quen (`≥4` lần/tháng). |
| **Q6** | Chia theo người (`453` train / `114` test, tránh thổi phồng `0,043` ROC-AUC của row split), **Random Forest** vượt `Dummy` (`0,500`) và `Logistic Regression` (`0,726`): đạt test **ROC-AUC = 0,782** (CI `[0,756; 0,808]`), **balanced accuracy = 0,705**, **accuracy = 72,3%** (vs `59,9%`). Lợi thế mạnh nhất ở xếp hạng xác suất (`>0,7`: **84,4%** nhận; `≤0,3`: **18,6%** nhận). Biến dẫn dắt: `coupon` (`0,157`), `CoffeeHouse` (`0,058`), `expiration` (`0,049`), `Bar` (`0,018`); nhân khẩu học xấp xỉ `0`. | `q6_fig1_model_comparison.png`, `q6_fig2_confusion_matrix.png`, `q6_fig3_errors_by_coupon.png`, `q6_fig4_permutation_importance.png` | Lỗi lệch mạnh theo lớp nền (`Carry out` FPR **90,5%**, `Bar` FNR **53,0%**); giả định biết trước thói quen (cold-start chưa giải quyết); chưa tối ưu ngưỡng theo chi phí FP/FN. |

## 3. Năm phát hiện chính & hạn chế tổng hợp của nhóm

### Năm phát hiện chính xuyên suốt Q1–Q6
1. **Loại coupon và thói quen tương ứng quyết định ý định nhận mạnh hơn nhân khẩu học (Q3, Q5, Q6):** Người thường ghé đúng loại địa điểm chấp nhận cao hơn tới **+39,58 điểm %** (`Bar`) và **+31,41 điểm %** (`Coffee House`), trong khi chênh lệch tuổi/thu nhập nhỏ hơn nhiều (`+7,5 pp` và `+4,4 pp`, thu nhập mất ý nghĩa khi cố định tuổi). Kết quả ML độc lập khẳng định điều này: 4 biến quan trọng nhất của Random Forest là `coupon`, `CoffeeHouse`, `expiration` và `Bar`, còn `age`/`income` đóng góp xấp xỉ `0`.
2. **Phân tích gộp thô dễ gây ngộ nhận do cấu trúc thiết kế kịch bản — phân tầng (D4) làm đảo chiều nhiều kết luận (Q1, Q2, Q4):**
   - *Q1:* `direction_same` thô chỉ chênh `+1,76 pp` nhưng trong `Home` tăng tới **+17,40 pp** vì toàn bộ `6.283` dòng `No Urgent Place` bị gắn `direction_same = 0`.
   - *Q2:* `10AM` gộp thô cho `2h > 1d` (`-2,01 pp`), nhưng tách theo từng coupon thì **cả 5/5 loại đều `1d > 2h`** (nghịch lý Simpson do tỷ trọng `Bar` ở `1d` là `36,1%` so với `5,4%` ở `2h`).
   - *Q4:* `Friend(s) - Alone` giảm từ `+14,76 pp` xuống còn **+1,33 pp** (CI chứa `0`) sau khi chuẩn hóa trên vùng common support vì `Friend(s)` không tồn tại trong kịch bản `Home`/`Work`.
3. **Độ nhạy với khoảng cách và thời hạn phụ thuộc rất mạnh vào tính chất dịch vụ (Q1, Q2):** `Carry out & Take away` giữ tỷ lệ nhận trên `72%` ngay cả khi đi `≥25` phút, còn `Restaurant(<20)` sụt **42,74 điểm %** (`75,32%` → `32,58%`). Tương tự, `1d` vượt trội ở ăn uống (`+22–24 pp`) và sáng/tối muộn, nhưng `Bar` lúc `10PM` có xu hướng nghiêng về `2h` (`50,25%` vs `41,38%`).
4. **Ràng buộc lặp người trả lời ảnh hưởng trực tiếp tới ước lượng bất định và đánh giá ML (Q5, Q6):** Dựng `respondent_id` từ hồ sơ cá nhân phát hiện `567` người (86,2% có đúng `22` kịch bản). Bootstrap theo dòng thu hẹp CI ~1,8 lần so với cluster-bootstrap theo người ở Q5; còn chia train/test ngẫu nhiên theo dòng làm ROC-AUC bị thổi phồng ảo `+0,043` so với chia theo người ở Q6.
5. **Mô hình dự đoán hữu ích nhất để xếp hạng xác suất hơn là phân loại cứng ở ngưỡng 0,5 (Q6):** Random Forest đạt ROC-AUC **0,782** trên `114` người test, phân tách tốt hai đầu xác suất (`>0,7`: **84,4%** nhận; `≤0,3`: **18,6%** nhận), nhưng sai nhiều ở vùng lưng chừng `0,3–0,6` và thiên lệch đoán “nhận” với coupon phổ biến (`Carry out` FPR `90,5%`).

### Hạn chế chung
- **Ý định giả định, không phải hành vi thực tế:** Khảo sát trên Amazon Mechanical Turk đo `Y` là ý định trả lời tình huống, chưa đo tỷ lệ đổi coupon thật hay doanh thu.
- **Không suy luận nhân quả:** Toàn bộ chênh lệch quan sát là mối liên hệ thống kê; biến chưa quan sát (tâm trạng, mức độ đói, giá món cụ thể) có thể còn gây nhiễu.
- **Khuyết thiếu `participant_id` gốc & ô rỗng thiết kế:** ID suy ra có thể gộp nhầm người trùng hồ sơ; một số giao điểm bối cảnh có `n=0` hoặc `n<30` khiến không thể khái quát cho mọi kịch bản.

## 4. Individual reflections *(từng thành viên tự điền trước khi nộp)*

- **Trần Nhật Dương:** *(Tự viết: khó khăn khi xử lý cấu trúc cờ `GEQ` và nghịch lý Simpson ở `10AM`/`direction_same`, bài học về module hóa & D2–D5, cách tự kiểm chứng khi dùng AI.)*
- **Nguyễn Mạnh Hà:** *(Tự viết: trải nghiệm kiểm soát vùng common support ở Q4, sensitivity analysis ở Q3, rà soát và tự kiểm chứng bản nháp.)*
- **Hoàng Công Lý:** *(Tự viết: bài học dựng `respondent_id` chống rò rỉ train/test, lý do chọn ROC-AUC thay vì F1, phân tích lỗi và cách dùng AI.)*

> **Lưu ý học thuật:** Theo quy định môn học, phần reflection cá nhân bắt buộc do từng sinh viên tự viết bằng trải nghiệm thực tế của mình.